# Information Extraction (Invoices Only)

This notebook develops and evaluates the **rule-based information extraction**
pipeline that runs after a document has been classified as `invoice` by the
classifier.

**Target fields** (per assignment):
- `invoice_number`
- `invoice_date`
- `due_date`
- `issuer`
- `recipient`
- `total`

## Input
We do **not** use `data/processed/full_dataset_preprocessed.csv` here — the
classification preprocessing lowercases text and strips numbers/dates, which
destroys everything IE needs. We read the raw invoice dataset instead:
`data/raw/invoices/converted_invoice_dataset.xlsx` (Invoice NER dataset from
Kaggle, 67 invoices). The `Input` column carries the raw invoice text and the
`Final_Output` column carries a JSON object with gold labels.

## Ground-truth coverage
The gold JSON uses field names such as `INVOICE_NUMBER`, `DATE`, `TOTAL`,
`ISSUED_TO`, `BILLED_TO`, etc. Coverage varies by invoice — not every field is
present in every row. We map those keys into our six-field schema before
scoring.

## Public API
```python
extract_invoice_fields(text: str) -> dict  # always returns all 6 keys, missing → None
```

## 0. Imports & paths

In [ ]:
import os
import re
import json
from pathlib import Path

import pandas as pd

if Path.cwd().name == 'notebooks':
    os.chdir('..')

import sys
sys.path.insert(0, str(Path.cwd()))

INVOICE_XLSX = Path('data/raw/invoices/converted_invoice_dataset.xlsx')

print('cwd:', Path.cwd())
print('invoice xlsx exists:', INVOICE_XLSX.exists())

## 1. Load the invoice dataset

Each row has the raw invoice text in `Input` and a JSON string of gold fields
in `Final_Output`. We keep the original casing, punctuation, and numbers — the
extractor needs all of it.

In [ ]:
df_inv = pd.read_excel(INVOICE_XLSX)
df_inv = df_inv.rename(columns={'Input': 'text', 'Final_Output': 'gold_json'})
df_inv['doc_id'] = [f'inv_{i:04d}' for i in range(len(df_inv))]
df_inv = df_inv[['doc_id', 'text', 'gold_json']]

print(f'Loaded {len(df_inv)} invoices')
print('\n--- sample ---')
print(df_inv.iloc[0]['text'][:500])

## 2. Parse ground-truth JSON

Each `Final_Output` value is a JSON dict whose keys vary per invoice. We map
those keys into our six-field schema. A field is considered unlabeled if none
of the candidate keys is present.

In [ ]:
GOLD_KEYS = {
    'invoice_number': ['INVOICE_NUMBER'],
    'invoice_date':   ['DATE', 'DATE_ISSUED', 'INVOICE_DATE', 'PAYMENT_DATE'],
    'due_date':       ['DUE_DATE', 'PAYMENT_DUE_DATE'],
    'total':          ['TOTAL_AMOUNT', 'TOTAL', 'TOTALS', 'GRAND_TOTAL'],
    'issuer':         ['ISSUER', 'COMPANY', 'FROM', 'VENDOR'],
    'recipient':      ['BILLED_TO', 'BILL_TO', 'ISSUED_TO', 'CUSTOMER_NAME', 'RECIPIENT'],
}


def _gold_for(gold: dict, field: str):
    for k in GOLD_KEYS[field]:
        if k in gold and gold[k]:
            return str(gold[k])
    return None


def parse_gold(json_str: str) -> dict:
    try:
        gold = json.loads(json_str)
    except Exception:
        return {k: None for k in GOLD_KEYS}
    return {k: _gold_for(gold, k) for k in GOLD_KEYS}


df_gt = df_inv[['doc_id']].copy()
for field in GOLD_KEYS:
    df_gt[f'gt_{field}'] = df_inv['gold_json'].apply(lambda s, f=field: _gold_for(json.loads(s) if isinstance(s, str) and s.strip() else {}, f))

print('Gold-label coverage:')
for field in GOLD_KEYS:
    n = df_gt[f'gt_{field}'].notna().sum()
    print(f'  {field:<16}: {n:>3d}/{len(df_gt)}')
df_gt.head()

## 3. Rule-based extractor

Imported from `src/information_extraction.py`, the single source of truth.
Editing the extractor there automatically updates both this notebook and the
live service. The public API is a single function that always returns all six
keys:

```python
extract_invoice_fields(text, pdf_bytes=None, image_bytes=None, words=None) -> dict
```

In [ ]:
from src.information_extraction import extract_invoice_fields, InvoiceFields

# quick smoke test on one doc
sample = df_inv.iloc[0]['text']
print(extract_invoice_fields(sample))

## 4. Run extractor on the full dataset

We apply `extract_invoice_fields` to every row. The function always returns a
dict with all six keys — missing values are `None` rather than raising — so it
is safe to use in a production pipeline.

In [ ]:
preds = df_inv['text'].apply(extract_invoice_fields).apply(pd.Series)
df_pred = pd.concat([df_inv[['doc_id']], preds], axis=1)

print('Field-level non-null rate:')
for col in ['invoice_number', 'invoice_date', 'due_date', 'issuer', 'recipient', 'total']:
    n = df_pred[col].notna().sum()
    print(f'  {col:<16}: {n:>3d}/{len(df_pred)}')

df_pred.head()

## 5. Evaluate against ground truth

For each field we compute accuracy over rows that have a gold label. Numeric
fields (`total`) and dates are normalized before comparison so that
equivalent values with different formatting (e.g. `1,234.00` vs `1234`) match.
Issuer/recipient use a substring-style match because the raw text often
carries minor punctuation/whitespace noise from the OCR step.

In [ ]:
def _norm_value(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ''
    return re.sub(r'[\s,$€£#]', '', str(s).lower()).rstrip('.')


def _norm_date(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ''
    return re.sub(r'[\s,.\-/]', '', str(s).lower())


def _norm_text(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ''
    return re.sub(r'[^a-z0-9]', '', str(s).lower())


def evaluate(df_pred: pd.DataFrame, df_gt: pd.DataFrame) -> pd.DataFrame:
    df = df_pred.merge(df_gt, on='doc_id', how='inner')
    field_norm = {
        'invoice_number': _norm_value,
        'invoice_date':   _norm_date,
        'due_date':       _norm_date,
        'total':          _norm_value,
        'issuer':         _norm_text,
        'recipient':      _norm_text,
    }
    results = []
    for field, norm_fn in field_norm.items():
        p = df[field].map(norm_fn)
        g = df[f'gt_{field}'].map(norm_fn)
        have_gt = g != ''
        exact = (p == g) & have_gt
        substr = pd.Series(
            [(a in b) or (b in a) if a and b else False for a, b in zip(p, g)],
            index=df.index,
        )
        relaxed = have_gt & (p != '') & substr
        results.append({
            'field': field,
            'n_with_gt': int(have_gt.sum()),
            'exact_match':   round(exact.sum() / max(have_gt.sum(), 1), 3),
            'relaxed_match': round(relaxed.sum() / max(have_gt.sum(), 1), 3),
        })
    return pd.DataFrame(results)


evaluate(df_pred, df_gt)

## 6. Error analysis

A handful of mismatches per field, so we know what to improve in the rules.

In [ ]:
def show_mistakes(df_pred, df_gt, field, n=5):
    df = df_pred.merge(df_gt, on='doc_id', how='inner')
    gt_col = f'gt_{field}'
    norm_fn = _norm_date if field in ('invoice_date', 'due_date') else (
        _norm_text if field in ('issuer', 'recipient') else _norm_value
    )
    mask = df[gt_col].notna() & (df[field].map(norm_fn) != df[gt_col].map(norm_fn))
    if not mask.any():
        print(f'[{field}] no mismatches')
        return
    for _, row in df[mask].head(n).iterrows():
        print(f"[{row['doc_id']}] pred={row[field]!r}  gt={row[gt_col]!r}")
    print()


for field in ['invoice_number', 'invoice_date', 'total', 'issuer', 'recipient']:
    print(f'--- {field} ---')
    show_mistakes(df_pred, df_gt, field)

## 7. Demo path — PDF in, fields out

For the live demo the input is a PDF. `src/pdf_loader.pdf_to_text` covers both
digitally-generated PDFs (via `pdfplumber`) and scanned PDFs (via `pypdfium2`
+ `pytesseract`). The same `extract_invoice_fields` function is used.

In [ ]:
from src.pdf_loader import pdf_to_text

# Replace with a real invoice PDF path to test end-to-end:
# pdf_path = 'path/to/invoice.pdf'
# pdf_bytes = Path(pdf_path).read_bytes()
# text = pdf_to_text(pdf_bytes)
# fields = extract_invoice_fields(text, pdf_bytes=pdf_bytes)
# print(fields)

# --- dry run on a dataset row (no PDF needed) ---
sample_text = df_inv.iloc[0]['text']
fields = extract_invoice_fields(sample_text)
print('doc_id :', df_inv.iloc[0]['doc_id'])
for k, v in fields.items():
    print(f'  {k:<16}: {v}')

## Status

| Item | Status |
|---|---|
| `src/information_extraction.py` — production extractor | ✅ done |
| `src/pdf_loader.py` — pdfplumber + OCR fallback | ✅ done |
| `src/preprocessing.py` — `clean_for_classifier()` | ✅ done |
| `src/service.py` — FastAPI `/classify` + `/extract` | ✅ done |
| Spatial-anchoring layout path (word bboxes) | ✅ done |
| Regex-text fallback path (`extract_from_text`) | ✅ done |